# Environment Setup & Connection Verification

In [1]:
import sys
import importlib.metadata
import langgraph
import langchain
import pinecone
import voyageai
import groq
import openai
import dotenv

print("Python:", sys.version)
print("Executable:", sys.executable)
print("-" * 50)

packages = ['langgraph', 'langchain', 'pinecone', 'voyageai', 'groq', 'openai', 'python-dotenv']
for pkg in packages:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")

print("\nAll core packages imported successfully.")

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Executable: d:\Hybrid-Rag\.venv\Scripts\python.exe
--------------------------------------------------
langgraph: 1.2.11
langchain: 1.3.15
pinecone: 9.1.0
voyageai: 0.5.0
groq: 1.6.0
openai: 3.0.0
python-dotenv: 1.2.2

All core packages imported successfully.


## 5.3 Test Voyage AI

In [2]:
import os
from dotenv import load_dotenv
import voyageai

load_dotenv()
VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")

vo = voyageai.Client(api_key=VOYAGE_API_KEY)
res = vo.embed(["Hello world"], model="voyage-3-lite")
print("Voyage AI connected! Embedding dim:", len(res.embeddings[0]))

Voyage AI connected! Embedding dim: 512


## 5.4 Test Pinecone

In [3]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

print("Connected to Pinecone.")
print(pc.list_indexes())

Connected to Pinecone.
IndexList([<name='hybrid-rag-dense', dim=1024, ready=True>])


## 5.5 Test Groq

In [4]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY_1")

groq_client = Groq(api_key=GROQ_API_KEY)

response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Groq connection successful."
        }
    ],
)

print(response.choices[0].message.content)

Groq connection successful.


## 5.6 Test OpenRouter

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY_1")

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

response = openrouter_client.chat.completions.create(
    model="meta-llama/llama-3.1-8b-instruct",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: OpenRouter connection successful."
        }
    ],
)

print(response.choices[0].message.content)

OpenRouter connection successful.


In [6]:
## testing ingestions pipeline : 
from src.ingestion import load_document

documents = load_document(
    "../data/raw/Langraph.docx"
)

print("Documents loaded:", len(documents))

print("\nFirst document:")
print(documents[0].page_content[:1000])

print("\nMetadata:")
print(documents[0].metadata)

Documents loaded: 1

First document:
LangGraph: The Comprehensive Guide

From Beginner to Advanced Concepts with Deep Theoretical Foundations



Table of Contents

Introduction to LangGraph
1.1 Why LangGraph?
1.2 The Actor Model and State Machines

Installation and Setup

Core Concepts
3.1 State as the Source of Truth
3.2 Nodes as Pure Functions (Almost)
3.3 Edges: Control Flow and Message Passing
3.4 Graphs: Directed, Possibly Cyclic

Your First Graph

State Management
5.1 Immutability and Reducers
5.2 TypedDict and Pydantic Schemas
5.3 Custom Reducers and Accumulation Patterns
5.4 Theoretical Underpinnings: Event Sourcing and CQRS

Nodes and Functions
6.1 Purity, Side Effects, and Determinism
6.2 Sync vs. Async Execution
6.3 Node Composition and Reusability

Edges and Control Flow
7.1 Normal Edges: Strict Sequencing
7.2 Conditional Edges: NonDeterminism in a Deterministic Shell
7.3 Parallel Execution and the Send API: ScatterGather Pattern

Building a Chatbot with MessageGraph
8.1 Th

In [7]:
## testing chunking :
from src.ingestion import load_document
from src.chunking import create_chunks

documents = load_document("../data/raw/Langraph.docx")

chunks = create_chunks(documents)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

Documents: 1
Chunks: 43


In [8]:
print("Documents:", len(documents))
print("Chunks:", len(chunks))

chunk_lengths = [
    len(chunk.page_content)
    for chunk in chunks
]

print("Smallest:", min(chunk_lengths))
print("Largest:", max(chunk_lengths))
print("Average:", sum(chunk_lengths) / len(chunk_lengths))

Documents: 1
Chunks: 43
Smallest: 254
Largest: 796
Average: 676.7209302325581


In [9]:
from src.embeddings import VoyageEmbedder

embedder = VoyageEmbedder()

vectors = embedder.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Number of vectors:", len(vectors))
print("Vector dimension:", len(vectors[0]))

Number of vectors: 43
Vector dimension: 1024


In [10]:
from src.vector_store import PineconeStore

vector_store = PineconeStore(
    index_name="hybrid-rag-dense"
)

In [11]:
records = []

for i, (chunk, vector) in enumerate(zip(chunks, vectors)):

    record = {
        "id": f"doc_0_chunk_{i}",
        "values": vector,
        "metadata": {
            "text": chunk.page_content,
            **chunk.metadata,
        },
    }

    records.append(record)

print("Records prepared:", len(records))

Records prepared: 43


In [12]:
vector_store.upsert(records)

print("Upsert completed.")

Upsert completed.


In [13]:
stats = vector_store.index.describe_index_stats()

print(stats)

DescribeIndexStatsResponse(dimension=1024, total_vector_count=43, metric='cosine', namespaces=1)


In [14]:
query="what is langgraph"
query_vector = embedder.embed_query(query)
results = vector_store.search(
    vector=query_vector,
    top_k=5,
)
for match in results["matches"]:
    print("\n---")
    print("ID:", match["id"])
    print("Score:", match["score"])
    print("Text:")
    print(match["metadata"]["text"][:500])


---
ID: doc_0_chunk_5
Score: 0.605708122
Text:
Under the hood, LangGraph is inspired by the Actor model and finitestate machines.

In the Actor model, independent actors (nodes) communicate only via messages (state updates). Each actor processes a message, may update its internal state, and can send messages to other actors. LangGraph’s nodes behave similarly: they receive the current state, optionally produce an update (a message), and the graph runtime routes it to the next node.

The graph itself is a state machine where the state is the 

---
ID: doc_0_chunk_3
Score: 0.567175925
Text:
Best Practices and Patterns

Troubleshooting

Conclusion



1. Introduction to LangGraph

LangGraph is a library for building stateful, multi-actor applications powered by Large Language Models (LLMs). It models your application as a directed graph—optionally cyclic—where each node is a function or agent that reads and writes a shared state, and edges define how execution flows from one node to anoth

In [15]:
from src.retrieval import DenseRetriever

retriever = DenseRetriever(
    index_name="hybrid-rag-dense",
    top_k=5,
)

results = retriever.retrieve(
    "What is persistence in langraph"
)

In [16]:
for rank, result in enumerate(results, start=1):
    print(f"\nRank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(result["metadata"]["text"])


Rank: 1
Score: 0.5475
9. Persistence and Checkpointing

9.1 Event Sourcing and the Checkpoint Model

LangGraph’s checkpointing is a form of event sourcing. After every node execution (a “step”), the runtime saves the current state as a checkpoint, along with metadata (node name, timestamp). The sequence of checkpoints forms an appendonly log. You can reconstruct the graph execution timeline by replaying the checkpoints.

The thread_id in the config provides multitenancy: each thread maintains its own independent sequence of checkpoints.

9.2 Storage Backends

MemorySaver: Inmemory, for development/testing. Not shared across processes.

SqliteSaver: Persistent singlefile storage. Good for local apps or demos.

PostgresSaver: For production, supports concurrent access and durability.

Rank: 2
Score: 0.4867
Loops: An agent may call tools multiple times until a condition is met.

Branching: The next step may depend on the LLM’s output or a human decision.

Pausing: A human may need to app

In [17]:

from src.sparse_retrieval import BM25Retriever


sparse_retriever = BM25Retriever(chunks)


query = "What is persistence ?"


results = sparse_retriever.retrieve(
    query,
    top_k=5,
)

for rank, result in enumerate(results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Rank: {rank}")
    print(f"Score: {result['score']:.4f}")
    print(result["text"])


Rank: 1
Score: 3.8328
updates: Emits only the state updates from each node. Best for showing what changed.

messages: For chat applications, emits tokens of the LLM’s response as they are generated. Each chunk is an AIMessageChunk, and metadata indicates which node produced it.

debug: Provides internal trace events (node start/end, checkpoint saves). Ideal for development.

Streaming fundamentally relies on the same checkpointing infrastructure; you can start streaming, pause, and later resume the stream.



11. Human-in-the-Loop

11.1 The Interrupt Mechanism as a Saga Breakpoint

Rank: 2
Score: 3.7327
10. Streaming

10.1 Reactive Streams and Backpressure

LangGraph supports streaming as a pullbased iterator (for chunk in graph.stream(...)) or pushbased async generator. However, it does not yet implement backpressure (i.e., the consumer cannot slow down the producer). This means for tokenlevel streaming, if the consumer is slow, tokens may buffer in memory. For most HTTPbased use cas

In [19]:
from src.retrieval import DenseRetriever
from src.sparse_retrieval import BM25Retriever

dense_retriever = DenseRetriever(
    index_name="hybrid-rag-dense",
    top_k=5,
)

sparse_retriever = BM25Retriever(chunks)

query = "what is persistence?"

dense_results = dense_retriever.retrieve(query)
sparse_results = sparse_retriever.retrieve(
    query,
    top_k=5,
)


In [ ]:
print("DENSE")
for rank, result in enumerate(dense_results, start=1):
    print(rank, result["id"])

print("\nSPARSE")
for rank, result in enumerate(sparse_results, start=1):
    print(rank, result["id"])


DENSE
1 doc_0_chunk_25
2 doc_0_chunk_4
3 doc_0_chunk_17
4 doc_0_chunk_26
5 doc_0_chunk_24

SPARSE
1 doc_0_chunk_28
2 doc_0_chunk_27
3 doc_0_chunk_8
4 doc_0_chunk_14
5 doc_0_chunk_35


In [21]:
from src.fusion import reciprocal_rank_fusion

fused_results = reciprocal_rank_fusion(
    [
        dense_results,
        sparse_results,
    ]
)

for rank, result in enumerate(fused_results, start=1):
    print(f"\n{'=' * 60}")
    print(f"RRF Rank: {rank}")
    print(f"ID: {result['id']}")
    print(f"RRF Score: {result['rrf_score']:.6f}")
    print(f"Original Score: {result['score']:.4f}")
    print(result["text"][:500])



RRF Rank: 1
ID: doc_0_chunk_25
RRF Score: 0.016393
Original Score: 0.4185
9. Persistence and Checkpointing

9.1 Event Sourcing and the Checkpoint Model

LangGraph’s checkpointing is a form of event sourcing. After every node execution (a “step”), the runtime saves the current state as a checkpoint, along with metadata (node name, timestamp). The sequence of checkpoints forms an appendonly log. You can reconstruct the graph execution timeline by replaying the checkpoints.

The thread_id in the config provides multitenancy: each thread maintains its own independent sequ

RRF Rank: 2
ID: doc_0_chunk_28
RRF Score: 0.016393
Original Score: 3.8328
updates: Emits only the state updates from each node. Best for showing what changed.

messages: For chat applications, emits tokens of the LLM’s response as they are generated. Each chunk is an AIMessageChunk, and metadata indicates which node produced it.

debug: Provides internal trace events (node start/end, checkpoint saves). Ideal for develop

In [22]:


from src.reranking import VoyageReranker



reranker = VoyageReranker()


query = "what is persistence?"


reranked_results = reranker.rerank(
    query=query,
    results=fused_results,
    top_k=5,
)




In [23]:

for rank, result in enumerate(reranked_results, start=1):


    print("\n" + "=" * 60)


    print(f"Rank: {rank}")
    print(f"ID: {result['id']}")
    print(f"Rerank Score: {result['rerank_score']:.4f}")


    print("\nTEXT:")
    print(result["text"][:800])


Rank: 1
ID: doc_0_chunk_4
Rerank Score: 0.6562

TEXT:
Loops: An agent may call tools multiple times until a condition is met.

Branching: The next step may depend on the LLM’s output or a human decision.

Pausing: A human may need to approve an action before continuing.

Persistence: If the system crashes, you should be able to resume exactly where you left off.

LangGraph addresses these by introducing cycles, builtin checkpointing, streaming, and a humanintheloop primitive. It bridges the gap between simple sequential chains and fullyfledged autonomous agents, while giving you finegrained control over every transition.

1.2 The Actor Model and State Machines

Under the hood, LangGraph is inspired by the Actor model and finitestate machines.

Rank: 2
ID: doc_0_chunk_25
Rerank Score: 0.6328

TEXT:
9. Persistence and Checkpointing

9.1 Event Sourcing and the Checkpoint Model

LangGraph’s checkpointing is a form of event sourcing. After every node execution (a “step”), the runtime saves

In [24]:
from src.retrieval import DenseRetriever
from src.sparse_retrieval import BM25Retriever
from src.reranking import VoyageReranker
from src.hybrid_retrieval import HybridRetriever

In [25]:
import importlib
import src.retrieval
import src.hybrid_retrieval

importlib.reload(src.retrieval)
importlib.reload(src.hybrid_retrieval)

from src.retrieval import DenseRetriever
from src.hybrid_retrieval import HybridRetriever


In [26]:
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=sparse_retriever,
    reranker=reranker,

    dense_top_k=10,
    sparse_top_k=10,
    rerank_top_k=5,
)

In [27]:

query = "what is persistence?"

results = hybrid_retriever.retrieve(
    query
)

In [28]:
for rank, result in enumerate(results, start=1):

    print("\n" + "=" * 60)

    print(f"Rank: {rank}")
    print(f"ID: {result['id']}")
    print(f"Rerank Score: {result['rerank_score']:.4f}")

    print("\nTEXT:")
    print(result["text"][:800])


Rank: 1
ID: doc_0_chunk_4
Rerank Score: 0.6562

TEXT:
Loops: An agent may call tools multiple times until a condition is met.

Branching: The next step may depend on the LLM’s output or a human decision.

Pausing: A human may need to approve an action before continuing.

Persistence: If the system crashes, you should be able to resume exactly where you left off.

LangGraph addresses these by introducing cycles, builtin checkpointing, streaming, and a humanintheloop primitive. It bridges the gap between simple sequential chains and fullyfledged autonomous agents, while giving you finegrained control over every transition.

1.2 The Actor Model and State Machines

Under the hood, LangGraph is inspired by the Actor model and finitestate machines.

Rank: 2
ID: doc_0_chunk_25
Rerank Score: 0.6328

TEXT:
9. Persistence and Checkpointing

9.1 Event Sourcing and the Checkpoint Model

LangGraph’s checkpointing is a form of event sourcing. After every node execution (a “step”), the runtime saves

LANGGRAPH IMPLEMENTATIONl

In [29]:
from src.llm import get_generation_llm

llm = get_generation_llm()

response = llm.invoke(
    "Explain persistence in LangGraph in two sentences."
)

print(response.content)

LangGraph achieves persistence by automatically saving the graph's state to a checkpointer (like SQLite or Postgres) after each step, creating a thread-specific history of execution. This allows workflows to be paused, resumed, replayed from specific points, or branched into alternative paths while maintaining full context and enabling human-in-the-loop intervention.


In [32]:
from src.graph import build_rag_graph
from src.llm import get_generation_llm

llm = get_generation_llm()

rag_graph = build_rag_graph(
    hybrid_retriever,
    llm,
)

result = rag_graph.invoke(
    {
        "query": "How do you implement the scatter-gather pattern for parallel execution using the Send API?"
    }
)

print("ANSWER:\n", result.get("answer"))


ANSWER:
 Based on the provided context, the scatter-gather pattern for parallel execution using the Send API is implemented as follows:

1. **Create a dispatcher node** that returns a list of `Send` objects, each targeting a worker node with a specific state shard:

```python
from langgraph.graph import Send

def dispatcher(state: State):
    return [Send("worker", {"item": i}) for i in state["items"]]
```

2. **Define a worker node** that processes each item independently:

```python
def worker(state: State):
    result = process(state["item"])
    return {"results": [result]}
```

3. **Declare the results field with an accumulator reducer** in your state definition to merge the parallel results back together (e.g., `lambda existing, new: existing + new`).

When the dispatcher returns a list of `Send` objects, the runtime fans out execution to the specified "worker" node with different state shards. Each parallel execution is independent and writes its own updates, which are merged ba